In [133]:
import sys
import os

sys.path.append(os.path.abspath("../")) 

import torch
from torch import nn
from blocks import CatDogDataset
from utils import parse_all_xmls
import json
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights

In [134]:
with open("../data/dataset_splits.json", "r") as f:
    dataset_splits = json.load(f)

batch_size = 64 
img_width = 448   
img_height = 448  
S = 7
B = 2 

train_dataset = CatDogDataset(dataset_splits["train"], "../data/images/", S, B, img_width, img_height)
val_dataset = CatDogDataset(dataset_splits["val"], "../data/images/", S, B, img_width, img_height)

train_loader = DataLoader(train_dataset, batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size, shuffle=False)

In [135]:
resnet = resnet18(weights=ResNet18_Weights.DEFAULT)

resnet

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [136]:
images, _ = next(iter(train_loader))

images.shape

torch.Size([64, 3, 448, 448])

In [137]:
resnet = nn.Sequential(*list(resnet.children())[:-2])

In [138]:
output = resnet(images)

output.shape

torch.Size([64, 512, 14, 14])

In [139]:
class DetectionHead(nn.Module):
    def __init__(self, in_channels, shrink_channels, expand_channels, S, B, num_classes):
        super().__init__()

        self.S = S
        self.B = B
        self.num_classes = num_classes

        self.shrink_conv1x1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=shrink_channels,
            kernel_size=(1, 1),
            stride=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(num_features=shrink_channels)

        self.conv3x3 = nn.Conv2d(
            in_channels=shrink_channels,
            out_channels=shrink_channels,
            kernel_size=(3, 3),
            stride=2,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(num_features=shrink_channels)

        self.expand_conv1x1 = nn.Conv2d(
            in_channels=shrink_channels,
            out_channels=expand_channels,
            kernel_size=(1, 1),
            stride=1,
            bias=False
        )

        self.bn3 = nn.BatchNorm2d(num_features=expand_channels)

        self.leaky_relu = nn.LeakyReLU(0.1)

        self.flat = nn.Flatten()

        self.dense1 = nn.Linear(
            in_features=expand_channels * S * S,
            out_features=4096
        )

        self.dropout = nn.Dropout(0.5)

        self.dense2 = nn.Linear(
            in_features=4096, 
            out_features= S * S * (num_classes + B * 5)
        )

    def forward(self, X):

        X = self.leaky_relu(self.bn1(self.shrink_conv1x1(X)))
        X = self.leaky_relu(self.bn2(self.conv3x3(X)))
        X = self.leaky_relu(self.bn3(self.expand_conv1x1(X)))

        X = self.flat(X)

        X = self.leaky_relu(self.dense1(X))
        X = self.dropout(X)
        out = self.dense2(X)

        return out.view(out.shape[0], self.S, self.S, -1)

In [140]:
CatDogNet = nn.Sequential(
    *list(resnet.children()),
    DetectionHead(
        in_channels=512, 
        shrink_channels=128, 
        expand_channels=1024,
        S=7,
        B=2,
        num_classes=len(train_dataset.classes)
    )
)

In [141]:
output = CatDogNet(images)

output.shape

torch.Size([64, 7, 7, 47])